In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, Configuração do Schema Inicial
USE CATALOG mvp_eng_dados;
USE SCHEMA gold;

# ANÁLISE DE DADOS
## Contextualizando as escolhas pelas perguntas de negócio

Como trabalho em uma instituição educacional de ensino superior, vários setores demonstram interesse de conhecimentos e respostas a partir dos nossos dados. Por isso, as perguntas de negócio foram divididas em categorias conforme área de interesse e setor como apresentado a seguir. As consultas foram reaalizadas baseando-se nos Data Marts criados para cada área na Camada Gold, considerando que já havia sido feito anteriormente um levantamento de perguntas com o setores e o acesso aos dados seria feito por múltiplos usuários. Os Data Marts pré-calculam as métricas para que os relatórios sejam super rápidos. Portanto, isso diminui o custo operacional da instituição, bem como ajuda no controle de necessidades e acessos aos relatórios (governança).

### Potencial de Conversão e Captação de Pós-Graduação (Setores: Marketing e Financeiro)

- Qual é a taxa de conversão (potencial_matricula_pos) por curso (nome_curso) e por área do conhecimento?
- Qual o tempo ideal após a graduação (meses_desde_formacao) em que os egressos demonstram maior intenção de matrícula?
- Ex-bolsistas de graduação (bolsista_graduacao) possuem maior probabilidade de continuar os estudos em comparação aos não bolsistas?

### Empregabilidade e Perfil Socioeconômico (Setores: Marketing e Financeiro)

- Qual é a renda média e a distribuição de cargos (nivel_cargo) por curso e por modalidade (modalidade_graduacao)?
- Existe correlação entre o valor da mensalidade da graduação (mensalidade_base) e o retorno financeiro atual do egresso (renda_mensal_estimada)?
- Quais cursos geram maior inserção na iniciativa privada vs. setor público vs. profissionais autônomos?

### Satisfação e Engajamento da Comunidade Alumni (Setores: Acadêmico e Marketing)

- Como o NPS da graduação (satisfacao_graduacao_nps) se relaciona com a pontuação de engajamento do ex-aluno (engajamento_alumni_score)?
- Promotores do curso (NPS 9-10) apresentam maior propensão a se matricular na pós-graduação do que detratores?
- A modalidade de ensino (EAD vs. Presencial) impacta o nível de engajamento pós-formação?

### Geografia e Oportunidades de Mercados Regionais (Setores: Acadêmico e Marketing)

- Quais estados (uf_residencia) concentram os egressos com maior potencial de matrícula para campanhas regionalizadas de marketing?
- Qual a distribuição espacial dos profissionais por área de atuação e renda média?

## 1. Potencial de Conversão e Captação de Pós-Graduação 
**_(Setores: Marketing e Financeiro)_**

### 1.1. Qual é a taxa de conversão (potencial_matricula_pos) por curso (nome_curso) e por área do conhecimento?
Análise Técnica: Cruzamos a tabela de fatos com a dimensão de cursos ou consultamos o Data Mart gold.kpi_potencial_pos_graduacao para calcular a proporção de egressos com interesse em pós-graduação sobre o total por curso e área.

Data Mart Utilizado: gold.kpi_potencial_pos_graduacao

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 1. Taxa de Conversão por Curso e Área do Conhecimento
SELECT 
    nome_curso,
    area_conhecimento,
    SUM(total_egressos) AS total_egressos,
    SUM(total_potencial_pos) AS total_potencial_pos,
    ROUND((SUM(total_potencial_pos) / SUM(total_egressos)) * 100, 2) AS taxa_conversao_pct
FROM gold.kpi_potencial_pos_graduacao
GROUP BY nome_curso, area_conhecimento
ORDER BY taxa_conversao_pct DESC;

### Análise dos resultados:

Com base nos dados apresentados no notebook para a Questão 1.1 (Taxa de Conversão por Curso e Área do Conhecimento), apresento a análise dos resultados:

Destaques de Desempenho

- Humanas Lidera o Ranking: Todos os 4 primeiros colocados pertencem à área de Humanas. Pedagogia ocupa o primeiro lugar com a maior taxa de conversão (18,34%), seguida de perto por Direito (17,99%), Psicologia (17,69%) e Administração (16,26%).

- Biológicas Apresenta Desempenho Consistente: Os cursos da área de Biológicas ocupam do 5º ao 8º lugar, mantendo uma taxa de conversão intermediária entre 12,69% (Nutrição) e 15,73% (Medicina Veterinária e Biomedicina).

- Tecnologia e Exatas com Baixa Proporção: Os cursos das áreas de Tecnologia e Exatas apresentaram as menores taxas de conversão (todas abaixo de 8%). Matemática Aplicada (5,86%) e Sistemas de Informação (5,68%) registraram os menores índices.

**_Insights para Estratégia de Negócio_**

Volume vs. Proporção: Cursos de Tecnologia como Ciência da Computação e ADS possuem a maior base de egressos (aproximadamente 1.000 alunos cada), porém baixíssima intenção proporcional de pós-graduação (aproximadamente 7%). O foco comercial para estes cursos deve ser em ofertas mais alinhadas ao mercado de trabalho (como certifições ou MBAs curtos) para tentar elevar o interesse.

Foco de Aquisição em Humanas e Saúde: Campanhas diretas de captação para programas de pós-graduação lato/stricto sensu têm maior eficiência e ROI quando direcionadas para egressos de Humanas e Biológicas, onde quase 1 em cada 5 alunos demonstra interesse ativo em continuar os estudos.

### 1.2. Qual o tempo ideal após a graduação (meses_desde_formacao) em que os egressos demonstram maior intenção de matrícula?
Análise Técnica: Agrupamos a intenção de matrícula pelas faixas de meses desde a formação (faixa_meses_formacao) já pré-calculadas na camada Gold para identificar a janela temporal de ouro para campanhas de captação.

Data Mart Utilizado: gold.kpi_conversao_tempo_bolsa

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 2. Janela Temporal Ideal de Conversão para Pós-Graduação
SELECT 
    faixa_meses_formacao,
    SUM(total_egressos) AS total_egressos,
    SUM(total_potencial_pos) AS total_potencial_pos,
    ROUND((SUM(total_potencial_pos) / SUM(total_egressos)) * 100, 2) AS taxa_intencao_pct
FROM gold.kpi_conversao_tempo_bolsa
GROUP BY faixa_meses_formacao
ORDER BY taxa_intencao_pct DESC;

### Análise dos resultados:
Com base no retorno da consulta SQL para a Questão 1.2 (Janela Temporal Ideal de Conversão para Pós-Graduação) no notebook, apresento a análise detalhada dos resultados:

Destaques do Desempenho por Faixa de Tempo

- Liderança Isolada (Mais de 36 meses): A maior propensão à pós-graduação está entre os egressos formados há mais de 3 anos, atingindo a maior taxa de intenção (11,59%) e concentrando a imensa maioria dos potenciais inscritos (978 de 1.236 alunos).

- Maturação Gradual da Intenção: Quanto mais tempo se passa desde a graduação, maior é a taxa de intenção de matrícula:
> - 0 a 12 meses: 6,95% de intenção
> - 13 a 24 meses: 7,32% de intenção
> - 25 a 36 meses: 9,30% de intenção
> - Mais de 36 meses: 11,59% de intenção

**_Insights e Recomendações Estratégicas_**

- O Momento da Decisão: Recém-formados (0 a 12 meses) possuem a menor taxa de conversão, possivelmente por focarem no ingresso imediato no mercado de trabalho ou por restrições financeiras. O interesse se consolida e cresce significativamente após o 2º ano de formado.

- Estratégia de Réguas de Relacionamento (Nutrição):
> - Anos 1 e 2: Manter engajamento contínuo com contéudos leves, networking, eventos da comunidade alumni e ofertas de cursos curtos de extensão.
> - Ano 3 em diante (Janela de Ouro): Intensificar campanhas comerciais diretas de captação para pós-graduação (lato e stricto sensu), aproveitando a fase em que o profissional busca especialização ou reposicionamento na carreira.

### 1.3. Ex-bolsistas de graduação (bolsista_graduacao) possuem maior probabilidade de continuar os estudos em comparação aos não bolsistas?
Análise Técnica: Realizamos o JOIN entre a fato e a dimensão gold.dim_egresso para comparar a taxa de propensão à pós-graduação entre alunos bolsistas e não bolsistas.

Data Mart Utilizado: gold.kpi_conversao_tempo_bolsa

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 3. Propensão à Pós-Graduação: Bolsistas vs. Não Bolsistas
SELECT 
    bolsista_graduacao,
    SUM(total_egressos) AS total_egressos,
    SUM(total_potencial_pos) AS total_potencial_pos,
    ROUND((SUM(total_potencial_pos) / SUM(total_egressos)) * 100, 2) AS taxa_intencao_pct
FROM gold.kpi_conversao_tempo_bolsa
GROUP BY bolsista_graduacao
ORDER BY taxa_intencao_pct DESC;

### Análise de resultados:
Com base nos resultados exibidos no notebook para a Questão 1.3 (Propensão à Pós-Graduação: Bolsistas vs. Não Bolsistas), apresento a análise detalhada dos dados:

Destaques do Desempenho

- Taxas de Propensão Praticamente Equivalentes: Não há diferença expressiva na intenção de matrícula entre os dois grupos.
> - Não Bolsistas (0): 10,90% de taxa de intenção.
> - Bolsistas (1): 10,33% de taxa de intenção.

- Volume Absoluto: Dos 1.236 potenciais alunos mapeados:
> - 635 são ex-bolsistas (de um total de 6.146 egressos bolsistas).
> - 601 são não bolsistas (de um total de 5.514 egressos não bolsistas).

**_Insights e Recomendações Estratégicas_**

- Mito Desmistificado: Ter sido bolsista durante a graduação não aumenta significativamente a probabilidade de continuar os estudos na pós-graduação em relação a quem pagou integralmente.

- Incentivos Financeiros: A leve diferença para baixo nos bolsistas (10,33% vs 10,90%) pode indicar maior sensibilidade a preço ou restrição orçamentária no pós-formação.

- Ação de Marketing/Vendas: Campanhas direcionadas ao público de ex-bolsistas devem priorizar ofertas com descontos progressivos, bolsas ex-aluno ou opções facilitadas de parcelamento para converter essa base expressiva em matrículas efetivas.

## 2. Empregabilidade e Perfil Socioeconômico
**_(Setores: Marketing e Financeiro)_**

### 2.1. Qual é a renda média e a distribuição de cargos (nivel_cargo) por curso e por modalidade (modalidade_graduacao)?
Análise Técnica: Intersecionamos os dados de curso, modalidade e emprego (gold.dim_emprego) para segmentar o avanço de carreira dos egressos e a sua respectiva renda.

Data Mart Utilizado: gold.kpi_empregabilidade_roi_curso

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 4. Renda Média Ponderada e Nível de Cargo por Curso e Modalidade
SELECT 
    nome_curso,
    modalidade_graduacao,
    nivel_cargo,
    SUM(qtd_egressos) AS total_egressos,
    ROUND(SUM(qtd_egressos * renda_media) / SUM(qtd_egressos), 2) AS renda_media_ponderada
FROM gold.kpi_empregabilidade_roi_curso
GROUP BY nome_curso, modalidade_graduacao, nivel_cargo
ORDER BY nome_curso, modalidade_graduacao, renda_media_ponderada DESC;